In [0]:
import requests
import os
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

# --- Variable ---
end_date = datetime.now()
start_date = end_date - timedelta(days=30)  

volume_path = "/Volumes/aws-s3-gh/default/databrick-ingestion-s3-gh-raw"
max_workers = 8

In [0]:
# --- Required Function ---
def get_gh_url(dt):
    # GH Archive URL Format: single digit for Hour only
    url_date_str = dt.strftime("%Y-%m-%d-%-H")
    gh_url = f"https://data.gharchive.org/{url_date_str}.json.gz"
    
    # Local Storage Folder Structure: /Year/Month/filename
    year_folder = dt.strftime("%Y")
    month_folder = dt.strftime("%m")
    filename = f"{url_date_str}.json.gz"
    
    # Builds: 2026/03/2026-3-3-9.json.gz
    relative_path = os.path.join(year_folder, month_folder, filename)
    
    return gh_url, relative_path

def download_file(dt):
    url, relative_path = get_gh_url(dt)
    # This combines /Volumes/... + /2026/03/filename
    full_target_path = os.path.join(volume_path, relative_path)
    
    # Extract the directory portion (/Volumes/.../2026/03)
    target_dir = os.path.dirname(full_target_path)
    
    if os.path.exists(full_target_path):
        return f"⏩ Skipped: {relative_path}"

    try:
        # Create the subfolders if not exist
        os.makedirs(target_dir, exist_ok=True)
        
        for attempt in range(3):
            with requests.get(url, stream=True, timeout=15) as r:
                if r.status_code == 200:
                    with open(full_target_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024*1024):
                            f.write(chunk)
                    return f"✅ Downloaded: {relative_path}"
                elif r.status_code == 404:
                    return f"⚠️ 404 Not Found: {relative_path}"
        return f"❌ Failed after retries: {relative_path}"
    except Exception as e:
        return f"💥 Error: {relative_path} -> {str(e)}"


In [0]:
# --- Download Process to AWS S3 ---
date_list = []
curr = start_date
while curr <= end_date:
    date_list.append(curr)
    curr += timedelta(hours=1)

print(f"🚀 Starting download of {len(date_list)} files...")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    results = list(executor.map(download_file, date_list))

for res in results:
    print(res)

In [0]:
table_checkpoint = volume_path + "/_checkpoints/bronze"
target_table = "`aws-s3-gh`.default.github_events"

# Define the Stream as Auto Loader for external file 
streaming_df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("pathGlobFilter", "*.json.gz")
  .option("cloudFiles.schemaLocation", f"{table_checkpoint}/schema")
  .load("/Volumes/aws-s3-gh/default/databrick-ingestion-s3-gh-raw"))

# 3. Write to the Table
# .awaitTermination() is key here to make the notebook wait for the data
query = (streaming_df.writeStream
  .option("checkpointLocation", f"{table_checkpoint}/data")
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .toTable(target_table))

query.awaitTermination()

# 4. Final verification
print(f"Ingestion complete. Current row count:")
display(spark.sql(f"SELECT count(*) FROM {target_table}"))